# Design matrix — one-camera setup

Single (left) camera variant of `../2_design_matrix.ipynb`.

Differences from the two-camera version:
- Only the left camera is loaded; both paws are read from `leftCamera`.
- Licks come from `get_left_licks` (left camera only) instead of `merge_licks`.
- No `avg_whisker_me` column (needs a second camera).
- One-camera videos can be recorded at **30 or 60 Hz**: the native rate is
  detected per session with `get_frame_rate`, the output grid is fixed at
  **fs = 30 Hz** for cross-session consistency, and an anti-aliasing low-pass
  is applied **only** when a session is downsampled (native > 30 Hz).

In [13]:
import os
import numpy as np
import pickle
import pandas as pd
from brainbox.io.one import SessionLoader
import gc
import concurrent.futures
import gzip

from segmentation_functions import get_left_licks, resample_common_time, interpolate_nans, low_pass, get_frame_rate
from one.api import ONE
one = ONE(mode='remote')

In [14]:
prefix = '/home/ines/repositories/'
prefix = '/Users/ineslaranjeira/Documents/Repositories/'

In [15]:
data_path = '/home/ines/repositories/representation_learning_variability/paper-individuality/segmentation/1_camera_setup/temp_data/design_matrices/' # data should be moved to the drive manually
data_path = prefix + 'representation_learning_variability/paper-individuality/data/design_matrices/1_camera_setup/session_1_individuality/' # data should be moved to the drive manually
data_path = "/Users/ineslaranjeira/Google Drive/O meu disco/CCU/PhD Project/paper-individuality/data/newly_generated/segmentation/design_matrices/training/video/"
# sessions = list(pd.read_csv('nm_filtered_eids.csv')['eid'])
sessions = list(pd.read_csv('lda_first_training_eids.csv')['eid'])

In [16]:

sessions_to_process = []
for m, mat in enumerate(sessions):
    file_path = one.eid2path(mat)
    
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]
    result_filename = data_path + "design_matrix_" + str(mat) + '_'  + mouse_name
    if not os.path.exists(result_filename):
        sessions_to_process.append(mat)

print(f"Found {len(sessions_to_process)} sessions to process.")

Found 43 sessions to process.


In [17]:
def process_design_matrix(session):

    file_path = one.eid2path(session)
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]
    try:
        
        """ LOAD VARIABLES """
        sl = SessionLoader(eid=session, one=one)
        sl.load_pose(views=['left'], tracker='lightningPose')
        # Load motion energy for the left camera only: load_session_data's default
        # (views=['left', 'right', 'body']) always raises ALFObjectNotFound on
        # 'right'/'body' for one-camera sessions, which load_session_data silently
        # swallows into a generic "Could not load motion_energy data" warning and
        # can leave sl.motion_energy without the 'leftCamera' key it needs.
        sl.load_motion_energy(views=['left'])
        # pupil=False: load_session_data's pupil step falls back to computing pupil
        # diameter on the fly, which internally calls self.load_pose(tracker='dlc')
        # (the default tracker, not lightningPose). That inner call resets self.pose
        # to {} before repopulating it, and if this session has no legacy 'dlc'
        # tracking (true for most one-camera/lightningPose-only sessions), it raises
        # before self.pose is restored -- silently wiping out the lightningPose data
        # we already loaded above. We don't use sl.pupil here, so just skip it.
        sl.load_session_data(trials=True, wheel=True, motion_energy=False, pupil=False)

        # Check if all data is available
        # if np.sum(sl.data_info['is_loaded']) >= 4:

        # Poses
        poses = sl.pose
        lc_t = np.asarray(poses['leftCamera']['times'])
        # One-camera videos can be recorded at 30 or 60 Hz; detect the native rate
        left_fr = get_frame_rate(lc_t)

        # Motion energy (anti-alias only when downsampling a 60 Hz session to the 30 Hz grid)
        me = sl.motion_energy
        mel_t = lc_t
        if left_fr > 30:
            motion_energy_l = low_pass(interpolate_nans(me['leftCamera']['whiskerMotionEnergy'], left_fr),
                                    cutoff=12, sf=left_fr)
        else:
            motion_energy_l = interpolate_nans(me['leftCamera']['whiskerMotionEnergy'], left_fr)

        # Licks
        features = ['tongue_end_l_x', 'tongue_end_l_y', 'tongue_end_r_x', 'tongue_end_r_y']
        lick_t, licks = get_left_licks(poses, features, common_fs=60)

        # Paws (a single camera sees both paws)
        if left_fr > 30:
            l_paw_x = low_pass(interpolate_nans(poses['leftCamera']['paw_r_x'], left_fr), cutoff=12, sf=left_fr)
            l_paw_y = low_pass(interpolate_nans(poses['leftCamera']['paw_r_y'], left_fr), cutoff=12, sf=left_fr)
            r_paw_x = low_pass(interpolate_nans(poses['leftCamera']['paw_l_x'], left_fr), cutoff=12, sf=left_fr)
            r_paw_y = low_pass(interpolate_nans(poses['leftCamera']['paw_l_y'], left_fr), cutoff=12, sf=left_fr)
        else:
            l_paw_x = interpolate_nans(poses['leftCamera']['paw_r_x'], left_fr)
            l_paw_y = interpolate_nans(poses['leftCamera']['paw_r_y'], left_fr)
            r_paw_x = interpolate_nans(poses['leftCamera']['paw_l_x'], left_fr)
            r_paw_y = interpolate_nans(poses['leftCamera']['paw_l_y'], left_fr)
        l_paw_t = lc_t
        r_paw_t = lc_t

        # Wheel
        wheel = sl.wheel
        wheel_t = np.asarray(wheel['times'], dtype=np.float64)
        wheel_vel = wheel['velocity'].astype(np.float32)

        # Common resampling window
        onset = max(lc_t.min(), wheel_t.min(), lick_t.min())
        offset = min(lc_t.max(), wheel_t.max(), lick_t.max())
        fs = 30
        ref_t = np.arange(onset, offset, 1 / fs, dtype=np.float64)

        # Restrict to time window
        def restrict(t, x):
            mask = (t >= onset) & (t <= offset)
            return t[mask], x[mask]

        mel_t, motion_energy_l = restrict(mel_t, motion_energy_l)
        wheel_t, wheel_vel = restrict(wheel_t, wheel_vel)
        l_paw_t_x, l_paw_x = restrict(l_paw_t, l_paw_x)
        l_paw_t_y, l_paw_y = restrict(l_paw_t, l_paw_y)
        r_paw_t_x, r_paw_x = restrict(r_paw_t, r_paw_x)
        r_paw_t_y, r_paw_y = restrict(r_paw_t, r_paw_y)
        lick_t, licks = restrict(lick_t, licks)

        # Resample
        mel_down, rt = resample_common_time(ref_t, mel_t, motion_energy_l, kind='linear')
        wh_down, _ = resample_common_time(ref_t, wheel_t, wheel_vel, kind='linear')
        lk_down, _ = resample_common_time(ref_t, lick_t, licks, kind='nearest')
        lpx_down, _ = resample_common_time(ref_t, l_paw_t_x, l_paw_x, kind='linear')
        lpy_down, _ = resample_common_time(ref_t, l_paw_t_y, l_paw_y, kind='linear')
        rpx_down, _ = resample_common_time(ref_t, r_paw_t_x, r_paw_x, kind='linear')
        rpy_down, _ = resample_common_time(ref_t, r_paw_t_y, r_paw_y, kind='linear')

        # Create design matrix
        design_matrix = pd.DataFrame({
            'Bin': rt,
            'Lick count': lk_down.astype(np.int8),
            'avg_wheel_vel': wh_down,
            'whisker_me': mel_down,
            'l_paw_x': lpx_down,
            'l_paw_y': lpy_down,
            'r_paw_x': rpx_down,
            'r_paw_y': rpy_down,
        })

        # """ LOAD TRIAL DATA """
        session_trials = sl.trials
        session_start = list(session_trials['goCueTrigger_times'])[0]
        design_matrix = design_matrix.loc[(design_matrix['Bin'] > session_start)]

        """ SAVE DATA """
        # Save unnormalized design matrix
        filename = data_path + "design_matrix_" + str(session) + '_'  + mouse_name
        design_matrix.to_parquet(filename, compression='gzip')

        # Save trials
        filename = data_path + "session_trials_" + str(session) + '_'  + mouse_name
        session_trials.to_parquet(filename, compression='gzip')

        del design_matrix, session_trials, sl
        gc.collect()

        # else:
        #     print('Data missing for session '+session)
            
    except Exception as e:
        # Print the full traceback (not just str(e)) so failures are diagnosable
        # instead of looking like generic "missing data" for the session.
        import traceback
        print(f"Failed on session {session}:")
        traceback.print_exc()


def parallel_process_data(sessions, function_name):
    with concurrent.futures.ThreadPoolExecutor() as executor:

        # Process each chunk in parallel
        executor.map(function_name, sessions)

In [18]:
for s, session in enumerate(sessions_to_process):
    process_design_matrix(session)

Failed on session 32088c28-be54-4e32-9f8e-a03c4f85c6d0:


Traceback (most recent call last):
  File "/var/folders/nt/d2j3zp9d1xzb8wgfrw81j0c40000gn/T/ipykernel_18548/1695351045.py", line 12, in process_design_matrix
    sl.load_pose(views=['left'], tracker='lightningPose')
  File "/Users/ineslaranjeira/Documents/Repositories/ibllib/brainbox/io/one.py", line 1391, in load_pose
    times_fixed, dlc = self._check_video_timestamps(view, pose_raw['times'], pose_raw[tracker])
KeyError: 'lightningPose'


2026-07-16 15:27:57 INFO     one.py:1288 Loading trials data


/opt/anaconda3/envs/iblenv/lib/python3.10/site-packages/one/util.py:428: ALFWarning: Multiple revisions: "", "2024-07-15"
  warnings.warn(f'Multiple revisions: {rev_list}', alferr.ALFWarning)


2026-07-16 15:27:58 INFO     one.py:1288 Loading wheel data
2026-07-16 15:27:59 INFO     one.py:1288 Loading trials data


...

2026-07-16 15:29:36 INFO     one.py:1288 Loading trials data
2026-07-16 15:29:37 INFO     one.py:1288 Loading wheel data
2026-07-16 15:29:39 INFO     one.py:1288 Loading trials data
2026-07-16 15:29:40 INFO     one.py:1288 Loading wheel data
2026-07-16 15:29:43 INFO     one.py:1288 Loading trials data
2026-07-16 15:29:44 INFO     one.py:1288 Loading wheel data
2026-07-16 15:29:46 INFO     one.py:1288 Loading trials data
2026-07-16 15:29:46 INFO     one.py:1288 Loading wheel data
2026-07-16 15:29:50 INFO     one.py:1288 Loading trials data
2026-07-16 15:29:50 INFO     one.py:1288 Loading wheel data
2026-07-16 15:29:53 INFO     one.py:1288 Loading trials data
2026-07-16 15:29:54 INFO     one.py:1288 Loading wheel data
2026-07-16 15:29:56 INFO     one.py:1288 Loading trials data
2026-07-16 15:29:56 INFO     one.py:1288 Loading wheel data
2026-07-16 15:29:59 INFO     one.py:1288 Loading trials data
2026-07-16 15:30:00 INFO     one.py:1288 Loading wheel data
2026-07-16 15:30:03 INFO     one

local md5 mismatch on dataset: churchlandlab_ucla/Subjects/UCLA052/2022-04-15/001/alf/_ibl_leftCamera.lightningPose.pqt
(S3) /Users/ineslaranjeira/Downloads/FlatIron/churchlandlab_ucla/Subjects/UCLA052/2022-04-15/001/alf/_ibl_leftCamera.lightningPose.pqt: 100%|██████████| 74.4M/74.4M [00:26<00:00, 2.81MB/s]


2026-07-16 15:30:37 INFO     one.py:1288 Loading trials data
2026-07-16 15:30:37 INFO     one.py:1288 Loading wheel data
2026-07-16 15:30:40 INFO     one.py:1288 Loading trials data
2026-07-16 15:30:40 INFO     one.py:1288 Loading wheel data
2026-07-16 15:30:42 INFO     one.py:1288 Loading trials data
2026-07-16 15:30:43 INFO     one.py:1288 Loading wheel data
2026-07-16 15:30:45 INFO     one.py:1288 Loading trials data
2026-07-16 15:30:46 INFO     one.py:1288 Loading wheel data
2026-07-16 15:30:49 INFO     one.py:1288 Loading trials data
2026-07-16 15:30:49 INFO     one.py:1288 Loading wheel data
Failed on session 114c0f58-fa7e-4b1c-901b-d201bb9a3b4a:


Traceback (most recent call last):
  File "/var/folders/nt/d2j3zp9d1xzb8wgfrw81j0c40000gn/T/ipykernel_18548/1695351045.py", line 12, in process_design_matrix
    sl.load_pose(views=['left'], tracker='lightningPose')
  File "/Users/ineslaranjeira/Documents/Repositories/ibllib/brainbox/io/one.py", line 1391, in load_pose
    times_fixed, dlc = self._check_video_timestamps(view, pose_raw['times'], pose_raw[tracker])
KeyError: 'lightningPose'


2026-07-16 15:30:54 INFO     one.py:1288 Loading trials data
2026-07-16 15:30:54 INFO     one.py:1288 Loading wheel data
2026-07-16 15:30:57 INFO     one.py:1288 Loading trials data
2026-07-16 15:30:57 INFO     one.py:1288 Loading wheel data
